# 04 - Kampanye tuning hyperparameter

Menjalankan grid yang dirancang di `tuning_grids/`. Prinsipnya human-in-the-loop:
tidak ada pencarian otomatis, urutan konfigurasi ditentukan manusia, dan setiap
baris hasil membawa kolom `catatan` yang merekam alasan konfigurasi itu dicoba.

Seleksi memakai split validation. Split test tidak disentuh sama sekali di
notebook ini.

Metode: grid kombinatorial untuk sumbu yang saling terkait, coordinate descent
untuk sumbu yang independen. Untuk RM-a, `lr`, `epochs`, dan `batch` bersama-sama
menentukan lintasan optimasi (batch 32 pada 5 epoch memberi separuh jumlah
langkah pembaruan dibanding batch 16), sehingga ketiganya harus digrid bersama;
`warmup_ratio` dan `weight_decay` efektif independen sehingga cukup dicoba satu
per satu di sel pemenang.

Prasyarat: `02_preprocessing.ipynb` sudah dijalankan.

In [1]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
runner.write_hardware()
print("device   :", runner.device)
print("keluaran :", runner.out_dir)

c:\Penelitian\IndoBERT-with-RAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-02 15:43:52,988 | INFO     | src.services.data | Data dimuat: train=6588 val=1402 test=1405 | device=cuda | encoder=indobenchmark/indobert-base-p2


device   : cuda
keluaran : C:\Penelitian\IndoBERT-with-RAC\outputs\tuning


## 1. Kalibrasi biaya

Jalankan satu konfigurasi RM-a lebih dulu untuk mengukur waktu dan memori
sesungguhnya di mesin ini, sebelum mempertaruhkan berjam-jam pada grid penuh.
Kalau memori kurang, turunkan `MICRO_BATCH` di `.env`; batch efektif tidak
berubah karena selisihnya ditutup akumulasi gradien.

In [2]:
kalibrasi = runner.run(
    "rma",
    {"lr": 2e-5, "epochs": 5, "batch": 16, "warmup_ratio": 0.1, "weight_decay": 0.01},
    note="kalibrasi biaya: baseline kanonik, sekaligus run #1 grid",
)

per_run = kalibrasi["train_time_s"]
print(f"satu run RM-a: {per_run:.0f} s | peak {kalibrasi['peak_mem_mb']:.0f} MB")
print(f"perkiraan 26 run RM-a: {per_run * 26 / 60:.0f} menit")

2026-09-02 15:43:56,483 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 1 run
2026-09-02 15:43:56,485 | INFO     | src.services.campaign | [rma] RUN #2 {'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 8, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7954.51it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-02 15:44:01,741 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-02 15:47:01,644 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9398
2026-09-02 15:49:50,858 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9647
2026-09-02 15:53:14,711 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9749
2026-09-02 15:56:49,261 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9740
2026-09-02 15:59:53,532 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9736
2026-09-02 15:59:54,083 | INFO     | src.services.campaign | [rma] RUN #2 val F1-macro 0.9749 (epoch terbaik 3)
satu run RM-a: 951 s | peak 2339 MB
perkiraan 26 run RM-a: 412 menit


## 2. Muat rancangan grid

In [3]:
GRID_DIR = settings.data_dir.parent / "tuning_grids"

def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]

for berkas in sorted(GRID_DIR.glob("*.csv")):
    print(f"  {berkas.name}: {len(pd.read_csv(berkas))} konfigurasi")

  RMA_TUNING_GRID.csv: 24 konfigurasi
  RMA_TUNING_GRID_STAGE2.csv: 2 konfigurasi
  RMB_TUNING_GRID.csv: 4 konfigurasi
  RMB_TUNING_GRID_STAGE1B.csv: 2 konfigurasi
  RMB_TUNING_GRID_STAGE2.csv: 15 konfigurasi
  RMB_TUNING_GRID_STAGE3.csv: 6 konfigurasi
  RMC_TUNING_GRID.csv: 66 konfigurasi
  RMC_TUNING_GRID_STAGE2.csv: 1 konfigurasi


## Melanjutkan kampanye yang terputus

`run_batch` menyaring konfigurasi yang sudah ada di riwayat secara default
(`resume=True`), sehingga sel batch di bawah aman dijalankan ulang apa adanya
setelah kernel mati, listrik padam, atau proses dihentikan. Yang sudah selesai
dilewati, penomoran run berlanjut, dan `best.json` tetap terjaga.

Yang hilang saat terputus hanyalah run yang sedang berjalan saat itu; run yang
sudah selesai ditulis atomik ke `runs_{skenario}.csv` begitu selesai.

Perbandingan memakai konfigurasi LENGKAP setelah nilai default diisi, dan untuk
RM-a `micro_batch` dinormalkan ke nilai efektifnya (`min(batch, micro_batch)`) —
nilai itulah yang menentukan ukuran batch di `DataLoader`.

Sel di bawah memperlihatkan apa yang tersisa sebelum batch dijalankan.

In [ ]:
def sisa(skenario: str, berkas: str) -> None:
    permintaan = muat_grid(berkas)
    tersisa = runner.pending_requests(skenario, permintaan)
    print(f"{berkas:34s} {len(permintaan) - len(tersisa):>3d}/{len(permintaan)} selesai, "
          f"{len(tersisa)} tersisa")

sisa("rma", "RMA_TUNING_GRID.csv")
sisa("rma", "RMA_TUNING_GRID_STAGE2.csv")
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    sisa("rmb", berkas)
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    sisa("rmc", berkas)

## 3. RM-a

Grid tahap 1 menggarap tiga sumbu yang saling terkait. Konfigurasi yang gagal
diisolasi ke `runs_rma_errors.csv` dan tidak menghentikan sisa antrean.

In [4]:
hasil_rma = runner.run_batch("rma", muat_grid("RMA_TUNING_GRID.csv"),
                             batch_id="rma_tahap1_grid")
hasil_rma.nlargest(10, "val_f1_macro")[
    ["run_id", "lr", "epochs", "batch", "val_f1_macro", "val_f1_judi",
     "train_time_s", "is_tie_with_best"]
]

2026-09-02 15:59:54,214 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 24 konfigurasi
2026-09-02 15:59:54,215 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 1/24 (perkiraan sisa 0.0 menit)
2026-09-02 15:59:54,246 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 2 run
2026-09-02 15:59:54,247 | INFO     | src.services.campaign | [rma] RUN #3 {'lr': 2e-05, 'epochs': 5, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9246.33it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-02 15:59:56,579 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-02 16:02:21,086 | INFO     | src.services.training | [RM-a] epoch 1/5 val F1-macro 0.9693
2026-09-02 16:04:54,921 | INFO     | src.services.training | [RM-a] epoch 2/5 val F1-macro 0.9678
2026-09-02 16:07:14,634 | INFO     | src.services.training | [RM-a] epoch 3/5 val F1-macro 0.9742
2026-09-02 16:09:35,555 | INFO     | src.services.training | [RM-a] epoch 4/5 val F1-macro 0.9738
2026-09-02 16:11:55,861 | INFO     | src.services.training | [RM-a] epoch 5/5 val F1-macro 0.9773
2026-09-02 16:11:56,711 | INFO     | src.services.run_log | Juara baru untuk rma: val F1-macro 0.9773 (sebelumnya 0.9749)
2026-09-02 16:12:06,456 | INFO     | src.services.campaign | [rma] RUN #3 val F1-macro 0.9773 (epoch terbaik 5)
2026-09-02 16:12:06,493 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 2/24 (perkiraan sisa 280.7 menit)
2026-09-02 16:12:06,528 | INFO     | src.services.run_log | 

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7797.12it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-02 16:12:08,561 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-02 16:14:28,652 | INFO     | src.services.training | [RM-a] epoch 1/3 val F1-macro 0.9593
2026-09-02 16:16:48,577 | INFO     | src.services.training | [RM-a] epoch 2/3 val F1-macro 0.9703
2026-09-02 16:19:08,755 | INFO     | src.services.training | [RM-a] epoch 3/3 val F1-macro 0.9703
2026-09-02 16:19:09,298 | INFO     | src.services.campaign | [rma] RUN #4 val F1-macro 0.9703 (epoch terbaik 2)
2026-09-02 16:19:09,332 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 3/24 (perkiraan sisa 211.8 menit)
2026-09-02 16:19:09,360 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 4 run
2026-09-02 16:19:09,362 | INFO     | src.services.campaign | [rma] RUN #5 {'lr': 2e-05, 'epochs': 8, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}


...

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7426.59it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-02 18:04:49,066 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-02 18:07:17,132 | INFO     | src.services.training | [RM-a] epoch 1/3 val F1-macro 0.9623
2026-09-02 18:09:50,729 | INFO     | src.services.training | [RM-a] epoch 2/3 val F1-macro 0.9679
2026-09-02 18:12:15,998 | INFO     | src.services.training | [RM-a] epoch 3/3 val F1-macro 0.9704
2026-09-02 18:12:16,859 | INFO     | src.services.campaign | [rma] RUN #13 val F1-macro 0.9704 (epoch terbaik 3)
2026-09-02 18:12:16,894 | INFO     | src.services.campaign | Batch rma_tahap1_grid: 12/24 (perkiraan sisa 156.4 menit)
2026-09-02 18:12:16,926 | INFO     | src.services.run_log | Riwayat runs_rma.csv dimuat: 13 run
2026-09-02 18:12:16,928 | INFO     | src.services.campaign | [rma] RUN #14 {'lr': 5e-05, 'epochs': 8, 'batch': 16, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'micro_batch': 32, 'seed': 42}


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10172.78it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-02 18:12:19,085 | INFO     | src.models.heads | Inisialisasi embedding special token: 3/3
2026-09-02 18:14:41,643 | INFO     | src.services.training | [RM-a] epoch 1/8 val F1-macro 0.9579


KeyboardInterrupt: 

Baca grid sebagai permukaan, bukan daftar. Heatmap `lr x epochs` per nilai
`batch` di `outputs/tuning/figures/` memperlihatkan apakah learning rate optimal
ikut bergeser saat batch berubah. Pemenang yang duduk di tepi grid adalah sinyal
untuk melebarkan rentang, bukan untuk langsung dikunci.

Selisih di bawah 0,15 pp dihitung seri karena hanya ada satu seed; pada kondisi
seri, pilih konfigurasi yang lebih murah.

In [ ]:
hasil_rma_tahap2 = runner.run_batch("rma", muat_grid("RMA_TUNING_GRID_STAGE2.csv"),
                                    batch_id="rma_tahap2_coordinate")
hasil_rma_tahap2[["run_id", "warmup_ratio", "weight_decay", "val_f1_macro",
                  "delta_vs_best_f1_macro_pp", "is_tie_with_best"]]

## 4. RM-b

In [ ]:
hasil_rmb = []
for berkas in ("RMB_TUNING_GRID.csv", "RMB_TUNING_GRID_STAGE1B.csv",
               "RMB_TUNING_GRID_STAGE2.csv", "RMB_TUNING_GRID_STAGE3.csv"):
    hasil_rmb.append(runner.run_batch("rmb", muat_grid(berkas),
                                      batch_id=berkas.replace(".csv", "").lower()))

pd.concat(hasil_rmb, ignore_index=True).nlargest(10, "val_f1_macro")[
    ["run_id", "head_arch", "hidden_dim", "lr", "epochs", "dropout",
     "val_f1_macro", "train_time_s", "trainable_params"]
]

## 5. RM-c

RM-c mewarisi head RM-b terbaik, jadi kampanye ini harus dijalankan SETELAH
RM-b selesai. Karena tidak ada training sama sekali, ratusan kombinasi
`alpha x k` selesai dalam hitungan detik.

In [ ]:
hasil_rmc = []
for berkas in ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv"):
    hasil_rmc.append(runner.run_batch("rmc", muat_grid(berkas),
                                      batch_id=berkas.replace(".csv", "").lower()))

pd.concat(hasil_rmc, ignore_index=True).nlargest(10, "val_f1_macro")[
    ["run_id", "alpha", "k", "weighting", "val_f1_macro", "val_f1_judi", "eval_time_s"]
]

## 6. Juara tiap skenario

In [ ]:
import json

best = json.loads((OUT_DIR / "best.json").read_text(encoding="utf-8"))
for skenario, entri in best.items():
    print(f"{skenario}: run #{entri['run_id']} | val F1-macro {entri['val_f1_macro']:.4f}")
    print(f"     {entri['config']}\n")

In [ ]:
summary = json.loads((OUT_DIR / "tuning_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Ringkasan

Seluruh angka di atas berasal dari split validation. Split test masih tertutup
dan baru dibuka satu kali di `05_final_benchmark.ipynb`.

Kalau ingin menambah konfigurasi setelah membaca hasil, panggil `runner.run`
atau `runner.run_batch` lagi: riwayat menumpuk dan penomoran run berlanjut.